# Setpoint Feedforward for an Ore Feed Conveyor

Setpoint feedforward control of the ore feed rate delivered by a long
conveyor belt, simulated with the python-control library. Adapted from the
GEL-7063 exercise `ex_anticipation_consigne.slx` (anticipation de la
consigne).

The control system is a 2-DOF structure with model-based setpoint
feedforward:

$$
U(s) = G_{ff}(s) R(s) + G_c(s) \left( G_r(s) R(s) - Y(s) \right)
$$

$$
Y(s) = G_p(s) U(s) + D(s)
$$

where:
 - $G_p(s)$ : plant (reclaim feeder and conveyor)
 - $G_m(s)$ : model of the plant, used to design $G_{ff}(s)$ and $G_r(s)$
 - $G_c(s)$ : PI feedback controller
 - $G_{ff}(s)$ : setpoint feedforward
 - $G_r(s)$ : reference model (desired setpoint response)
 - $D(s)$ : output disturbance

In [ ]:
from pathlib import Path

import control as con
import matplotlib.pyplot as plt
import numpy as np

plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

## Plant

The conveyor belt is loaded by a variable speed reclaim feeder. To allow
for the momentum of the feeder, its response to the speed setpoint $u$ is
modelled as a second-order lag with time constant $\tau_f = 3$ s. The conveyor is $L = 90$ m long and runs at
a fixed speed of $v = 1$ m/s, so ore loaded onto the belt reaches the
discharge after a transport delay of $\theta = L / v = 90$ s. The output
$y$ is the ore feed rate at the conveyor discharge:

$$
G_p(s) = \frac{K_p}{(\tau_f s + 1)^2} e^{-\theta s}
= \frac{1}{(3s + 1)^2} e^{-90s}
$$

Signals are in normalized units, so the gain is $K_p = 1$.

python-control transfer functions are rational, so the time delay is
kept separate from the rational part $G_{p,0}(s)$. For simulation, the
models are converted to discrete time, where the delay is represented
exactly as a shift of $\theta / T_s$ samples (see
[Discretization](#Discretization)).

In [ ]:
s = con.tf('s')

# Plant parameters
K_p = 1.0      # gain (normalized units)
tau_f = 3.0    # feeder time constant (s), second-order lag
L = 90.0       # conveyor length (m)
v = 1.0        # conveyor speed (m/s)
theta = L / v  # transport delay (s)

# Rational part of plant
G_p0 = K_p / (tau_f * s + 1) ** 2
print(G_p0)

In [ ]:
# Step response of the plant (the rational part shifted by the delay)
t_step = np.linspace(0, 150, 1501)
y_step = con.step_response(G_p0, T=t_step).outputs
y_step = np.interp(t_step - theta, t_step, y_step, left=0.0)

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(t_step, y_step)
ax.set_xlabel('Time (s)')
ax.set_ylabel('$y(t)$')
ax.set_title('Plant step response')
ax.grid()
plt.tight_layout()
plt.savefig(plot_dir / 'conveyor_plant_step.png', dpi=150)
plt.show()

## Discretization

All blocks are converted to discrete time with a zero-order hold (ZOH)
at sample time $T_s$. The time delay is then an exact shift of
$n_d = \theta / T_s$ samples:

$$
G_p(z) = G_{p,0}(z) \, z^{-n_d}
$$

The other blocks are discretized in the same way where they are defined
below.

The ZOH conversion is exact for blocks whose input is constant between
samples (e.g. the setpoint path when $r(t)$ steps at a sample time). In
the feedback loop the results are an approximation of the continuous
system, with errors that decrease with $T_s$ (less than 0.01 in $y(t)$ for
$T_s = 0.5$ s).

In [ ]:
# Sample time
Ts = 0.5

# Time delay as a shift of n_d samples
n_d = round(theta / Ts)
assert np.isclose(n_d * Ts, theta), "delay must be a multiple of Ts"
z_delay = con.ss(con.tf([1], [1] + [0] * n_d, Ts))

# Discrete-time plant (ZOH)
G_p_d = z_delay * con.ss(con.c2d(G_p0, Ts, 'zoh'))
print(f"Delay: {n_d} samples")

## Closed-Loop System

The function below constructs the closed-loop system from its component
transfer functions. Each block is given named input and output signals,
and the blocks are connected with `con.interconnect`:

```none
         ┌─────────┐ u_ff
     ┌──►│ G_ff(s) │──────────────────────────────┐
     │   └─────────┘                              │                     d
     │                                            │                     │
     │                                            │                     │
     │   ┌─────────┐ r_m      e ┌────────┐ u_fb   ▼+ u  ┌────────┐ y_p  ▼+
r ───┴──►│ F_r(s)  │─────►(Σ)──►│ G_c(s) │──────►(Σ)───►│ G_p(s) │────►(Σ)──┬──► y
         └─────────┘     + ▲−   └────────┘      +       └────────┘    +     │
                           │                                                │
                           └────────────────────────────────────────────────┘
```

Only the feedback controller $G_c$ and the plant $G_p$ are required. The
other blocks are optional:

| Argument | Block | Default |
|---|---|---|
| `F_r` | Reference filter, $r \rightarrow r_m$ | 1 (no filter) |
| `G_ff` | Setpoint feedforward, $r \rightarrow u_{ff}$ | `None` (no feedforward) |

The disturbance $d$ is added directly to the plant output.

For the setpoint feedforward design below, the reference filter is the
reference model, $F_r = G_r$.

In [ ]:
def make_closed_loop_system(G_c, G_p, F_r=1, G_ff=None, name='closed_loop'):
    """Construct a closed-loop system with optional setpoint feedforward.

    All systems must have the same sample time as G_p (or all be
    continuous-time). Scalars are treated as static gains.

    Args:
        G_c: Feedback controller, e -> u_fb.
        G_p: Plant, u -> y_p.
        F_r: Reference filter, r -> r_m (default 1, no filter).
        G_ff: Setpoint feedforward, r -> u_ff (default None, no
            feedforward).
        name: Name of the closed-loop system.

    Returns:
        Closed-loop system with inputs (r, d) and outputs
        (r_m, e, u, y), where d is an output disturbance.
    """
    dt = G_p.dt

    def block(sys, inputs, outputs, name):
        if np.isscalar(sys):
            sys = con.tf(sys, 1, dt)
        return con.ss(sys, inputs=inputs, outputs=outputs, name=name)

    blocks = [
        block(F_r, 'r', 'r_m', 'F_r'),
        block(G_c, 'e', 'u_fb', 'G_c'),
        block(G_p, 'u', 'y_p', 'G_p'),
        con.summing_junction(inputs=['r_m', '-y'], output='e', dt=dt, name='sum_e'),
        con.summing_junction(inputs=['y_p', 'd'], output='y', dt=dt, name='sum_y'),
    ]
    if G_ff is None:
        u_inputs = ['u_fb']
    else:
        blocks.append(block(G_ff, 'r', 'u_ff', 'G_ff'))
        u_inputs = ['u_ff', 'u_fb']
    blocks.append(
        con.summing_junction(inputs=u_inputs, output='u', dt=dt, name='sum_u')
    )

    return con.interconnect(
        blocks, inputs=['r', 'd'], outputs=['r_m', 'e', 'u', 'y'], name=name
    )

## Simulation

Simulation parameters:

| Parameter | Value |
|---|---|
| Simulation time | 0 – 1200 s |
| Setpoint step $r(t)$ | 0 → 1 at $t = 10$ s |
| Disturbance step $d(t)$ | 0 → 1 at $t = 600$ s |
| Initial conditions | zero |

The discrete-time closed-loop systems are simulated at the sample times
$t = k T_s$.

In [ ]:
# Simulation parameters
t_stop = 1200.0
t_r, r_step = 10.0, 1.0     # setpoint step time and size
t_d, d_step = 600.0, 1.0    # disturbance step time and size

t = Ts * np.arange(round(t_stop / Ts) + 1)
r = np.where(t >= t_r - Ts / 2, r_step, 0.0)
d = np.where(t >= t_d - Ts / 2, d_step, 0.0)


def simulate(T):
    """Simulate closed-loop system T with the inputs (r, d).

    Returns:
        Dictionary of output signals (r_m, e, u, y).
    """
    sim = con.forced_response(T, T=t, U=[r, d])
    return {name: sim.outputs[T.find_output(name)] for name in T.output_labels}

In [ ]:
def plot_sim_results(t, r, sim, title=None, filename=None):
    """Plot the setpoint, output, error and control signals, and save
    the figure to plot_dir if a filename is given."""
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 1 + 3*1.5))

    ax = axes[0]
    ax.plot(t, sim['y'], label='$y(t)$')
    ax.plot(t, r, label='$r(t)$')
    ax.plot(t, sim['r_m'], '--', label='$r_m(t)$')
    ax.set_ylabel('$r(t)$, $y(t)$')
    ax.set_title(title)
    ax.legend()

    ax = axes[1]
    ax.plot(t, sim['e'])
    ax.set_ylabel('$e(t)$')

    ax = axes[2]
    ax.plot(t, sim['u'])
    ax.set_ylabel('$u(t)$')
    ax.set_xlabel('Time (s)')

    for ax in axes:
        ax.grid()
    plt.tight_layout()
    if filename is not None:
        plt.savefig(plot_dir / filename, dpi=150)
    plt.show()

## PI Feedback Controller

$$
G_c(s) = K_c \frac{T_i s + 1}{T_i s}
$$

The controller is tuned with the improved SIMC rule (Skogestad and
Grimholt, 2012). The plant is first approximated by a first-order plus
time delay model $k e^{-\theta' s} / (\tau_1 s + 1)$ using Skogestad's half
rule: half of the second time constant is added to the first time
constant and half to the time delay,

$$
\tau_1 = \tau_f + \frac{\tau_f}{2}, \qquad \theta' = \theta + \frac{\tau_f}{2}
$$

The improved rule replaces $\tau_1$ by $\tau_1 + \theta'/3$ in the SIMC PI
rule, which gives better performance for delay-dominant processes:

$$
K_c = \frac{1}{k} \frac{\tau_1 + \theta'/3}{\tau_c + \theta'}, \qquad
T_i = \min \left( \tau_1 + \theta'/3, \; 4(\tau_c + \theta') \right)
$$

The closed-loop time constant $\tau_c$ is the tuning parameter. The
recommended choice $\tau_c = \theta'$ gives a robust, smooth response.

Reference: S. Skogestad and C. Grimholt (2012), "The SIMC method for
smooth PID controller tuning", in *PID Control in the Third Millennium*,
Springer.

In [ ]:
# First-order plus time delay approximation (half rule)
tau_1 = tau_f + tau_f / 2    # time constant (s)
theta_1 = theta + tau_f / 2  # time delay (s)

# Improved SIMC tuning
tau_c = theta_1  # closed-loop time constant (s)
K_c = (tau_1 + theta_1 / 3) / (K_p * (tau_c + theta_1))
T_i = min(tau_1 + theta_1 / 3, 4 * (tau_c + theta_1))
print(f"K_c = {K_c:.4f}, T_i = {T_i:.1f} s")

G_c = K_c * (T_i * s + 1) / (T_i * s)
G_c_d = con.c2d(G_c, Ts, 'zoh')

### Simulation 1: Feedback Control Only

The closed-loop system with the PI controller only: no reference filter
($F_r = 1$) and no setpoint feedforward.

In [ ]:
T_fb = make_closed_loop_system(G_c_d, G_p_d, name='feedback_only')
sim_fb = simulate(T_fb)
plot_sim_results(t, r, sim_fb, title='Feedback control only',
                 filename='conveyor_feedback_only.png')

## Reference Filter

The reference filter $F(s)$ specifies the desired setpoint tracking
dynamics (user choice). Since the invertible part of the plant model is
second order, a second-order filter is needed to make the feedforward
proper:

$$
F(s) = \frac{1}{(T_F s + 1)^2}, \qquad T_F = 10 \text{ s}
$$

In [ ]:
# Reference filter parameters
T_F = 10.0     # filter time constant (s)

F = 1 / (T_F * s + 1) ** 2
F_d = con.c2d(F, Ts, 'zoh')
print(F)

### Simulation 2: Reference Filter

The setpoint is filtered by $F(s)$ before the error is computed
($F_r = F$), with no setpoint feedforward.

In [ ]:
T_filt = make_closed_loop_system(G_c_d, G_p_d, F_r=F_d, name='ref_filter')
sim_filt = simulate(T_filt)
plot_sim_results(t, r, sim_filt, title='Reference filter',
                 filename='conveyor_ref_filter.png')

## Setpoint Feedforward and Reference Model

The setpoint feedforward is designed from the reference filter $F(s)$
and a model of the plant.

### Plant Model Decomposition

The model of the plant is assumed to be perfect, $G_m(s) = G_p(s)$. It
is decomposed into invertible and non-invertible parts:

$$
G_m(s) = \tilde{G}_m(s) \bar{G}_m(s)
= \frac{1}{(3s + 1)^2} \cdot e^{-90s}
$$

The non-invertible part $\bar{G}_m(s)$ is the transport delay. As before,
the delay is kept separate from the rational part, which here is
$\bar{G}_{m,0}(s) = 1$.

In [ ]:
# Model of the plant (perfect model, rational part)
G_m0 = G_p0

# Invertible part
G_m_inv = K_p / (tau_f * s + 1) ** 2

# Non-invertible part (rational part)
G_m_ninv0 = con.tf([1], [1])

# Check that the two parts recombine to the model rational part
diff = con.minreal(G_m_inv * G_m_ninv0 - G_m0, verbose=False)
assert np.allclose(diff.num[0][0], 0.0)
print(G_m_inv)
print(G_m_ninv0)

### Feedforward and Reference Model

The feedforward controller and the resulting tracking dynamics are:

$$
G_{ff}(s) = \frac{F(s)}{\tilde{G}_m(s)} = \frac{(3s + 1)^2}{(10s + 1)^2}
$$

$$
G_r(s) = F(s) \bar{G}_m(s) = \frac{1}{(10s + 1)^2} e^{-90s}
$$

$G_r(s)$ is the reference model: the setpoint response obtained with the
feedforward alone when the model is perfect, $G_r(s) = G_{ff}(s) G_m(s)$.
The feedback controller acts on the error between this reference
trajectory and the plant output.

In [ ]:
# Feedforward (minreal cancels any common factors)
G_ff = con.minreal(F / G_m_inv, verbose=False)
print(G_ff)

In [ ]:
# Reference model (rational part)
G_r0 = F * G_m_ninv0
print(G_r0)

In [ ]:
# Check that G_ff * G_m0 equals the reference model rational part
diff = con.minreal(G_ff * G_m0 - G_r0, verbose=False)
assert np.allclose(diff.num[0][0], 0.0)
print(con.minreal(G_ff * G_m0, verbose=False))

### Discretization of the Feedforward

The ZOH discretization of a product of transfer functions is not the
product of their ZOH discretizations. If $G_{ff}(s)$ and $G_r(s)$ were
discretized independently, $G_{ff}(z) G_m(z)$ would differ slightly from
$G_r(z)$ and the error $e(t)$ would be small but non-zero even with a
perfect model.

Forming the discrete feedforward as $F(z) / \tilde{G}_m(z)$ avoids this
but is not a good option here: the ZOH discretization of the second-order
$\tilde{G}_m(s)$ has a sampling zero on the negative real axis (near
$z = -0.89$), which would become a pole of the feedforward and cause an
oscillating feedforward signal.

Instead, $G_{ff}(s)$ is discretized directly, and the discrete reference
model is defined as the feedforward path through the discrete model of
the plant:

$$
G_r(z) = G_{ff}(z) G_m(z)
$$

so that the reference trajectory is exactly the output of the model when
driven by the feedforward. It is very close to the ZOH discretization of
the continuous $G_r(s)$.

In [ ]:
# Discrete-time feedforward (ZOH)
G_ff_d = con.c2d(G_ff, Ts, 'zoh')

# Discrete-time model of the plant and reference model
G_m_d = z_delay * con.ss(con.c2d(G_m0, Ts, 'zoh'))
G_r_d = G_m_d * con.ss(G_ff_d)

# Compare with the ZOH discretization of the continuous reference model
t_check = Ts * np.arange(round(200 / Ts) + 1)
y_r = con.step_response(G_r_d, T=t_check).outputs
y_F = con.step_response(con.ss(con.c2d(G_r0, Ts, 'zoh')) * z_delay,
                        T=t_check).outputs
print(G_ff_d)
print(f"Max difference from c2d(G_r): {np.abs(y_r - y_F).max():.4f}")

### Simulation 3: Setpoint Feedforward

The reference model is used as the reference filter ($F_r = G_r$), with
the setpoint feedforward $G_{ff}$.

In [ ]:
T_ff = make_closed_loop_system(
    G_c_d, G_p_d, F_r=G_r_d, G_ff=G_ff_d, name='setpoint_ff'
)
sim_ff = simulate(T_ff)
plot_sim_results(t, r, sim_ff, title='Setpoint feedforward',
                 filename='conveyor_setpoint_ff.png')

### Simulation 4: Plant-Model Mismatch in the Time Delay

The feedforward $G_{ff}$ and reference model $G_r$ are designed with the
model delay $\theta = 90$ s. Here the actual plant delay $\theta_p$ is
shorter (70 s) or longer (110 s) than the model delay, e.g. due to a change
in conveyor speed. The feedforward then no longer produces the reference
trajectory exactly, so the error $e(t)$ is non-zero during the setpoint
change and the feedback controller has to act.

In [ ]:
def make_discrete_plant(theta_p):
    """Discrete-time plant (ZOH) with the rational part G_p0 and time
    delay theta_p (must be a multiple of Ts)."""
    n = round(theta_p / Ts)
    assert np.isclose(n * Ts, theta_p), "delay must be a multiple of Ts"
    delay = con.ss(con.tf([1], [1] + [0] * n, Ts))
    return delay * con.ss(con.c2d(G_p0, Ts, 'zoh'))


# Actual plant delays (s)
plant_delays = [70.0, theta, 110.0]

sims_mm = {}
for theta_p in plant_delays:
    T_mm = make_closed_loop_system(
        G_c_d, make_discrete_plant(theta_p), F_r=G_r_d, G_ff=G_ff_d,
        name='setpoint_ff_mismatch'
    )
    sims_mm[rf'$\theta_p = {theta_p:g}$ s'] = simulate(T_mm)

In [ ]:
def plot_sim_comparison(t, r, sims, title=None, filename=None):
    """Plot the output, error and control signals of several
    simulations, given as a dictionary {label: sim}, and save the
    figure to plot_dir if a filename is given."""
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 1 + 3*1.5))

    ax = axes[0]
    ax.plot(t, r, 'k-', lw=1, label='$r(t)$')
    for label, sim in sims.items():
        ax.plot(t, sim['y'], label=label)
    ax.plot(t, sim['r_m'], 'k--', lw=1, label='$r_m(t)$')
    ax.set_ylabel('$r(t)$, $y(t)$')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize='small')

    for label, sim in sims.items():
        axes[1].plot(t, sim['e'], label=label)
        axes[2].plot(t, sim['u'], label=label)
    axes[1].set_ylabel('$e(t)$')
    axes[2].set_ylabel('$u(t)$')
    axes[2].set_xlabel('Time (s)')

    for ax in axes:
        ax.grid()
    plt.tight_layout()
    if filename is not None:
        plt.savefig(plot_dir / filename, dpi=150)
    plt.show()


plot_sim_comparison(t, r, sims_mm, title='Plant-model mismatch in the time delay',
                    filename='conveyor_delay_mismatch.png')

### Simulation 5: Plant-Model Mismatch with SIMC PID Control

Simulation 4 is repeated with a PID feedback controller tuned with the
SIMC rules for a second-order plus time delay process (Skogestad, 2003):

$$
g(s) = \frac{k}{(\tau_1 s + 1)(\tau_2 s + 1)} e^{-\theta s}
$$

$$
K_c = \frac{1}{k} \frac{\tau_1}{\tau_c + \theta}, \qquad
\tau_I = \min \left\{ \tau_1, \; 4(\tau_c + \theta) \right\}, \qquad
\tau_D = \tau_2
$$

These settings are for the series (interactive) form of the PID
controller, in which the PI and PD parts are multiplied.
A first-order filter with time constant $\alpha \tau_D$ is added to the
derivative term to make the controller proper:

$$
G_c(s) = K_c \frac{\tau_I s + 1}{\tau_I s} \cdot
\frac{\tau_D s + 1}{\alpha \tau_D s + 1}
$$

For the feeder, $\tau_1 = \tau_2 = \tau_f = 3$ s and $\theta = 90$ s, and
$\tau_c = \theta$ is used as before. The controller then cancels both
plant poles. The setpoint feedforward and reference model are unchanged.

Reference: S. Skogestad (2003), "Simple analytic rules for model reduction
and PID controller tuning", *Journal of Process Control*, 13, 291–309.

In [ ]:
# SIMC PID tuning (series form)
tau_c_pid = theta  # closed-loop time constant (s)
K_c_pid = tau_f / (K_p * (tau_c_pid + theta))
tau_I = min(tau_f, 4 * (tau_c_pid + theta))
tau_D = tau_f
alpha = 0.1        # derivative filter factor
print(f"K_c = {K_c_pid:.4f}, tau_I = {tau_I:.1f} s, tau_D = {tau_D:.1f} s")

G_c_pid = (
    K_c_pid * (tau_I * s + 1) / (tau_I * s)
    * (tau_D * s + 1) / (alpha * tau_D * s + 1)
)
G_c_pid_d = con.c2d(G_c_pid, Ts, 'zoh')
print(G_c_pid)

In [ ]:
sims_mm_pid = {}
for theta_p in plant_delays:
    T_mm = make_closed_loop_system(
        G_c_pid_d, make_discrete_plant(theta_p), F_r=G_r_d, G_ff=G_ff_d,
        name='setpoint_ff_mismatch_pid'
    )
    sims_mm_pid[rf'$\theta_p = {theta_p:g}$ s'] = simulate(T_mm)

plot_sim_comparison(t, r, sims_mm_pid,
                    title='Plant-model mismatch in the time delay, SIMC PID',
                    filename='conveyor_delay_mismatch_pid.png')

With the PID controller, there is a small derivative kick in $u(t)$ when
the output disturbance occurs at $t = 600$ s, limited by the derivative
filter.

On a near-pure-delay process like this, derivative action buys almost
nothing. That is consistent with the finding of Grimholt and Skogestad
(2018) that for pure time delay processes the optimal PID controller is a
PI controller.

Reference: C. Grimholt and S. Skogestad (2018), "Should we forget the
Smith Predictor?", *IFAC-PapersOnLine*, 51(4), 769–774.